## Análises de Negócio - Logística e Satisfação
Este *notebook* responde às perguntas propostas na fase de planeamento utilizando consultas na camada Gold.

### Uso do schema

In [0]:
catalog = "mvp_data_engeneering"
schema = "gold"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

### 1. Visão Geográfica: Rotas Críticas
Qual é a taxa média de atraso agrupada por Estado (UF) de origem e destino?

In [0]:
# Exibe as rotas interestaduais com maior média de dias de atraso (apenas rotas com mais de 50 pedidos)
spark.sql("""
    SELECT 
        uf_vendedor AS Origem, 
        uf_cliente AS Destino, 
        COUNT(order_id) AS total_pedidos,
        ROUND(AVG(dias_atraso), 1) AS media_dias_atraso,
        ROUND(SUM(is_atrasado) / COUNT(order_id) * 100, 1) AS percentagem_pedidos_atrasados
    FROM fato_logistica
    WHERE uf_vendedor IS NOT NULL AND uf_cliente IS NOT NULL
    GROUP BY uf_vendedor, uf_cliente
    HAVING total_pedidos > 50
    ORDER BY media_dias_atraso DESC
    LIMIT 10
""").display()

### 2. Localização do Estrangulamento (Gargalo)
Onde ocorre a maior perda de tempo logístico: no processamento ou no transporte?

In [0]:
# Compara os dias gastos pelo vendedores (processamento) com a transportadora (trânsito)
spark.sql("""
    SELECT 
        ROUND(AVG(dias_processamento), 1) AS media_dias_processamento_interno,
        ROUND(AVG(dias_transporte), 1) AS media_dias_transporte_transportadora,
        ROUND(AVG(dias_processamento + dias_transporte), 1) AS tempo_total_medio_logistico
    FROM fato_logistica
""").display()

### 3. Impacto na Satisfação do Cliente
Qual é a correlação direta entre o atraso e a nota de avaliação?

In [0]:
# Analisa como a percentagem de pedidos atrasados afeta diretamente as avaliações
spark.sql("""
    SELECT 
        review_score AS nota_avaliacao,
        COUNT(order_id) AS volume_avaliacoes,
        ROUND(AVG(dias_atraso), 1) AS media_dias_atraso,
        ROUND(SUM(is_atrasado) / COUNT(order_id) * 100, 1) AS percentagem_pedidos_atrasados
    FROM fato_logistica
    WHERE review_score IS NOT NULL
    GROUP BY review_score
    ORDER BY review_score ASC
""").display()

### 4. Performance de Parceiros (Vendedores)
Quais são os vendedores que falham sistematicamente no prazo de expedição (mais de 10 dias de processamento)?

In [0]:
# Identifica os piores vendedores no que toca a processamento interno (mínimo de 30 pedidos)
spark.sql("""
    SELECT 
        seller_id,
        uf_vendedor,
        COUNT(order_id) AS total_pedidos,
        ROUND(AVG(dias_processamento), 1) AS media_dias_processamento
    FROM fato_logistica
    GROUP BY seller_id, uf_vendedor
    HAVING total_pedidos > 30
    ORDER BY media_dias_processamento DESC
    LIMIT 10
""").display()